# Lab 06 solution, notebook 1: give the shopping list a third tool

The worked answer to the challenge of `06a_meet_mcp.ipynb`. The setup repeats the notebook's client, tool
conversion and `ask` loop, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import json
import os
import sys
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


## The server copy with `remove_item`

The copy is `solved/shopping_server_plus.py`. Its first two tools are the ones in `shopping_server.py`. We print
the third.

In [2]:
SERVER_FILE = ROOT / "06_MCP" / "solved" / "shopping_server_plus.py"

source = SERVER_FILE.read_text()
print(source[source.index("@mcp.tool()\ndef remove_item"):source.index("if __name__")].rstrip())

@mcp.tool()
def remove_item(item: str) -> str:
    """Remove one item from the shopping list."""
    items = list_items()
    if item not in items:
        return f"{item} is not on the list."
    items.remove(item)
    LIST_FILE.write_text("".join(line + "\n" for line in items), encoding="utf-8")
    return f"Removed {item}."


`remove_item` reads the list with `list_items()`, which is still a plain Python function inside the server file.
If the item is there, it removes the first line that matches and writes the file again, one item per line. Either
way it returns a sentence, so the model always has something to pass on.

The server finds its list from its own location, so this copy keeps a list of its own, in `solved/outputs/`. We
start it with three items.

In [3]:
LIST_FILE = SERVER_FILE.parent / "outputs" / "shopping_list.txt"
LIST_FILE.parent.mkdir(exist_ok=True)
LIST_FILE.write_text("milk\neggs\nbread\n", encoding="utf-8")

print(LIST_FILE.read_text(encoding="utf-8").splitlines())

['milk', 'eggs', 'bread']


## 1. List the copy's tools and call the new one by hand

The client is the one from the notebook, pointed at the copy.

In [4]:
from mcp import Client, StdioServerParameters

SERVER = StdioServerParameters(command=sys.executable, args=[str(SERVER_FILE)])

async with Client(SERVER) as mcp_client:
    listed = await mcp_client.list_tools()

print([tool.name for tool in listed.tools])
print(json.dumps(listed.tools[2].input_schema, indent=2))

['add_item', 'list_items', 'remove_item']
{
  "type": "object",
  "properties": {
    "item": {
      "title": "Item",
      "type": "string"
    }
  },
  "required": [
    "item"
  ],
  "title": "remove_itemArguments"
}


Three tools. The schema of `remove_item` has one required string, `item`, built from `item: str` like the schema
of `add_item`.

A tool is tested by hand before a model uses it. We add an item, remove it again, and try to remove one that was
never there.

In [5]:
async with Client(SERVER) as mcp_client:
    for name, arguments in [
        ("add_item", {"item": "jam"}),
        ("remove_item", {"item": "jam"}),
        ("remove_item", {"item": "caviar"}),
    ]:
        result = await mcp_client.call_tool(name, arguments)
        print(name, arguments, "->", result.content[0].text, "| isError:", result.is_error)

print(LIST_FILE.read_text(encoding="utf-8").splitlines())

add_item {'item': 'jam'} -> Added jam. | isError: False
remove_item {'item': 'jam'} -> Removed jam. | isError: False
remove_item {'item': 'caviar'} -> caviar is not on the list. | isError: False
['milk', 'eggs', 'bread']


`Added jam.`, `Removed jam.`, and `caviar is not on the list.` The last call is not an error: `isError` is
`False`, because the tool ran and gave a correct answer. The file is back to milk, eggs and bread.

## 2. Ask the model to remove an item

`to_chat_tool` and `ask` are copied from the notebook without a change. `ask` asks the server for its tools on
every question, so it picks up `remove_item` by itself.

In [6]:
from openai import OpenAI, omit

client = OpenAI(timeout=60)


def to_chat_tool(tool):
    """Turn one MCP tool into the tool description Chat Completions expects."""
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": tool.input_schema,
        },
    }


async def ask(question, max_turns=5):
    async with Client(SERVER) as mcp_client:
        listed = await mcp_client.list_tools()
        tools = [to_chat_tool(tool) for tool in listed.tools]
        messages = [{"role": "user", "content": question}]

        for turn in range(max_turns):
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
                reasoning_effort=EFFORT or omit,
            )
            reply = response.choices[0].message
            if not reply.tool_calls:
                return reply.content

            messages.append(reply)
            for call in reply.tool_calls:
                arguments = json.loads(call.function.arguments)
                result = await mcp_client.call_tool(call.function.name, arguments)
                text = "\n".join(block.text for block in result.content)
                print("SERVER CALLED:", call.function.name, arguments)
                messages.append({"role": "tool", "tool_call_id": call.id, "content": text})

    return "No answer after " + str(max_turns) + " turns."

In [7]:
print(await ask("I bought the eggs. Take them off my shopping list and tell me what is left."))
print()
print("file:", LIST_FILE.read_text(encoding="utf-8").splitlines())

SERVER CALLED: remove_item {'item': 'eggs'}


SERVER CALLED: list_items {}


I removed the eggs. What’s left on your shopping list: milk and bread.

file: ['milk', 'bread']


Two `SERVER CALLED` lines: `remove_item` with `{'item': 'eggs'}`, then `list_items`. The answer says the eggs are
gone and that milk and bread are left, and the file agrees: `['milk', 'bread']`.

**Notice** the argument. We wrote *the eggs*, and the model sent `eggs`, the word as it stands in the list. Our tool
compares exact text, so `Eggs` or `the eggs` would have found nothing. The model had not read the list, so it
guessed the spelling, and it guessed right. A model that calls `list_items` first reads the exact spelling in the
result.

## 3. Ask it to remove something that is not on the list

In [8]:
print(await ask("Take the caviar off my shopping list."))
print()
print("file:", LIST_FILE.read_text(encoding="utf-8").splitlines())

SERVER CALLED: remove_item {'item': 'caviar'}


Caviar isn’t on your shopping list, so I couldn’t remove it.

file: ['milk', 'bread']


One `SERVER CALLED` line, and the tool's sentence, *caviar is not on the list*, comes back in the model's own
words. The file did not change. Because the tool returns a sentence in both cases, the model has a fact to report
and does not have to guess what an empty result means.

## What changed, and what did not

One file changed: the server gained a function with a docstring and a typed parameter. The client, `to_chat_tool`
and `ask` are the notebook's code, untouched, and the model started using the new tool on the next question. A new
tool is added in the server, and every program that uses the server gets it.